In [1]:
using Cosmology
using Jens
using Jens.LensModel.ComLens: CombinedLens
using Jens.LensModel: SIS, Shear
using Jens.LightModel: PointImage
using Jens.LensGenerator: LensedPlane, LightPlane, GenGrid
using Jens.LensSystem: ForwardModel
using Jens.TimeDelay: LensTimeDelay, image_time_delays
using Jens.LensSolver: solve_images
using Jens.LensConstants: DAY_TO_SEC
using Jens.LensPointLikelihood: point_image_chi2, time_delay_chi2
using Jens.LensAdaptiveGrid: RefinementMap, adaptive_grid_info
using Statistics

## lens system

In [2]:
cosmo = Cosmology.FlatLCDM(0.7, 0.3, 0.0, 0.0)
z_lens, z_src = 0.3, 1.5
beta_x, beta_y = 0.05, -0.03

lens = CombinedLens(
    SIS   => (theta_E=1.0, xcentre=0.0, ycentre=0.0),
    Shear => (gamma1=0.05, gamma2=-0.02, xcentre=0.0, ycentre=0.0),
)
lp = LensedPlane(lens; z_lens=z_lens, cosmology=cosmo)

grid = GenGrid(pix_n=128, pix_size=0.09)
agn = PointImage(flux=100.0, beta_x=beta_x, beta_y=beta_y)
sys = ForwardModel(
    lens_plane   = lp,
    source_plane = LightPlane(agn; z=z_src),
    grid         = grid,
);

In [3]:
images, delays = image_time_delays(sys, beta_x, beta_y)

println("Found $(length(images)) images:")
for (i, (tx, ty, mu)) in enumerate(images)
    println("  $i: θ=($(round(tx, digits=4)), $(round(ty, digits=4)))  " *
            "μ=$(round(mu, digits=1))  Δt=$(round(delays[i] / DAY_TO_SEC, digits=3)) days")
end
println("  Δt(1↔2) = $(round(abs(delays[1] - delays[2]) / DAY_TO_SEC, digits=3)) days")

Found 2 images:
  1: θ=(0.8422, -0.2997)  μ=7.1  Δt=-31.316 days
  2: θ=(-0.3363, 0.6622)  μ=-11.0  Δt=-22.745 days
  Δt(1↔2) = 8.57 days


### Exact match → χ² ≈ 0

In [4]:
# Extract observed positions: discard magnification
obs_positions = [(tx, ty) for (tx, ty, mu) in images]

# Position χ² — observed uncertainty 0.003 arcsec
chi2_pos = point_image_chi2(images, obs_positions, 0.003)
println("χ²(position) = $chi2_pos  (expect 0)")

# Time-delay χ² — uncertainty 0.5 days
obs_pairs = [(1, 2, delays[1] - delays[2])]
chi2_dt = time_delay_chi2(delays, obs_pairs, 0.5 * DAY_TO_SEC)
println("χ²(delay)   = $chi2_dt  (expect 0)")

χ²(position) = 0.0  (expect 0)
χ²(delay)   = 0.0  (expect 0)


### Wrong model → χ² > 0

In [5]:
for theta_E_wrong in [0.8, 0.9, 0.95, 1.0, 1.05, 1.1, 1.2]
    lens_wrong = CombinedLens(
        SIS   => (theta_E=theta_E_wrong, xcentre=0.0, ycentre=0.0),
        Shear => (gamma1=0.05, gamma2=-0.02, xcentre=0.0, ycentre=0.0),
    )
    lp_w = LensedPlane(lens_wrong; z_lens=z_lens, cosmology=cosmo)
    sys_w = ForwardModel(
        lens_plane   = lp_w,
        source_plane = LightPlane(agn; z=z_src),
        grid         = grid,
    )
    imgs_w, delays_w = image_time_delays(sys_w, beta_x, beta_y)

    if length(imgs_w) != length(images)
        χ² = Inf
    else
        χ² = point_image_chi2(imgs_w, obs_positions, 0.003)
    end
    println("  θ_E=$theta_E_wrong  →  χ² = $χ²")
end

  θ_E=0.8  →  χ² = 6433.676160811518
  θ_E=0.9  →  χ² = 1606.3798996085523
  θ_E=0.95  →  χ² = 401.099122963885
  θ_E=1.0  →  χ² = 0.0
  θ_E=1.05  →  χ² = 399.8134662393917
  θ_E=1.1  →  χ² = 1596.3106982561362
  θ_E=1.2  →  χ² = 6360.275508138979


### 1d. Full MCMC example

Fit θ_E and external shear to the observed image positions.
This is the simplest H₀-ready point-source MCMC.

In [6]:
using Jens.LensMH: lens_mh

# Define the logp closure
function make_logp(obs_pos, obs_dt_pairs, sigma_pos, sigma_dt, beta_x, beta_y)
    return function logp(params)
        theta_E, gamma1, gamma2 = params

        m = CombinedLens(
            SIS   => (theta_E=theta_E, xcentre=0.0, ycentre=0.0),
            Shear => (gamma1=gamma1, gamma2=gamma2, xcentre=0.0, ycentre=0.0),
        )
        lp_m = LensedPlane(m; z_lens=z_lens, cosmology=cosmo)
        sys_m = ForwardModel(
            lens_plane   = lp_m,
            source_plane = LightPlane(agn; z=z_src),
            grid         = grid,
        )

        imgs_m, dlays_m = image_time_delays(sys_m, beta_x, beta_y)

        chi2 = point_image_chi2(imgs_m, obs_pos, sigma_pos)
        isinf(chi2) && return -Inf

        if !isempty(obs_dt_pairs)
            chi2 += time_delay_chi2(dlays_m, obs_dt_pairs, sigma_dt)
        end

        return -0.5 * chi2
    end
end

# Run MCMC — recover known parameters
logp = make_logp(obs_positions, obs_pairs, 0.003, 0.5*DAY_TO_SEC, beta_x, beta_y)
lo = [0.5, -0.1, -0.1]
hi = [1.5,  0.1,  0.1]

result = lens_mh(logp, lo, hi; n=2000, adapt=true)
samples = result.samples  # (3, N)

println("Parameter recovery (burn-in: first 500 discarded):")
for (i, name) in enumerate(["theta_E", "gamma1", "gamma2"])
    truth = [1.0, 0.05, -0.02][i]
    chain = samples[i, 501:end]
    println("  $name:  median=$(round(median(chain), digits=4))  " *
            "±$(round(std(chain), digits=4))  (truth=$truth)")
end

Parameter recovery (burn-in: first 500 discarded):
  theta_E:  median=1.0003  ±0.0028  (truth=1.0)
  gamma1:  median=0.0499  ±0.0005  (truth=0.05)
  gamma2:  median=-0.0201  ±0.0006  (truth=-0.02)
